weather forecast ML. Version 3. Forecast incluted. Madrid

Data collection

In [ ]:
#imports
import pandas as pd
import numpy as np
import urllib.request
from datetime import datetime, timedelta
import pytz
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import requests
from tqdm import tqdm
import time


In [ ]:
#getting the data from the Iowa Environmental Mesonet (IEM) API regarding (Station) City #Add station name
# FIXING THE DATE: Get exactly yesterday in UTC. 6 years in total
utc_now = datetime.now(pytz.utc)
end_date = (utc_now - timedelta(days=1)).date()
start_date = end_date.replace(year=end_date.year - 6)

# Construct the IEM API URL for City (Station)#Add station name
# We are requesting:
# tmpf = Air Temperature (Fahrenheit)
# dwpf = Dew Point (Fahrenheit)
# drct = Wind Direction (Degrees)
# sknt = Wind Speed (Knots)
# NEW API URL: Added 'p01i' (Rain) and 'skyc1' (Clouds)
# Construct the IEM API URL for City (Station)
url = (
    f"https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?"
    f"station=##STATION##&data=tmpf&data=dwpf&data=drct&data=sknt&" # <--- Add station
    f"data=skyc1&data=alti&"
    f"year1={start_date.year}&month1={start_date.month}&day1={start_date.day}&"
    f"year2={end_date.year}&month2={end_date.month}&day2={end_date.day}&"
    f"tz=Etc/UTC&format=onlycomma&latlon=no&missing=empty"
)

print(f"Downloading 6 years of data from: {url}")
print("This might take 10-20 seconds depending on server load...")

# Read the data directly from the URL into a pandas DataFrame
df_raw = pd.read_csv(url, parse_dates=['valid'])

# Rename the columns so they are easier for us to read
df_raw.rename(columns={
    'valid': 'Timestamp',
    'tmpf': 'Temp_F',
    'dwpf': 'Dew_Point_F',
    'drct': 'Wind_Dir_Deg',
    'sknt': 'Wind_Speed_Kts',
    'skyc1': 'Cloud_Cover_Code'
}, inplace=True)


# 2. Fix the Clouds: Map text codes to numerical Oktas (0 to 8)
cloud_map = {'CLR': 0, 'SKC': 0, 'FEW': 2, 'SCT': 4, 'BKN': 6, 'OVC': 8, 'VV ': 8}
# If the sensor reports nothing, we assume it's clear (0)
df_raw['Cloud_Cover_Okta'] = df_raw['Cloud_Cover_Code'].map(cloud_map).fillna(0)

# Drop the old text column
df_raw.drop(columns=['Cloud_Cover_Code'], inplace=True)

# Clean out rows missing the core temperature data
df_raw = df_raw.dropna(subset=['Temp_F', 'Wind_Dir_Deg', 'Dew_Point_F'])

# Localize Timezone
df_raw['Timestamp'] = df_raw['Timestamp'].dt.tz_localize('UTC').dt.tz_convert('Continent/City') #   <---- Add Continent/City

print("\n--- Full Physics Data Ready ---")
print(f"Total clean rows: {len(df_raw):,}")
display(df_raw.head())




Forecasting data engineering


GFS surfuce tempratures

In [ ]:
# Install the European binary decoder (System Level)
!apt-get install -y libeccodes0

#  Install the Python data engineering stack
!pip install herbie-data xarray cfgrib tqdm

from herbie import Herbie
import xarray as xr
from tqdm import tqdm

In [ ]:
# City Coordinates (STATION) #Add city/station
lat = # Add latitude
lon_gfs =   #Add longitude CRITICAL: Check if we should use the 360 Longitude Math
#Define the 5-Year Window
#  Anchor strictly to UTC
utc_now = datetime.now(pytz.utc)
#  Subtract 2 days for the AWS Government indexing buffer, and strip to pure dates
end_date = (utc_now - timedelta(days=2)).date()
#  Look back exactly 5 years for Compute Optimization
start_date = end_date.replace(year=end_date.year - 5)
date_list = pd.date_range(start=start_date, end=end_date, freq='D')

# The Forecast Hours to pull from the 00z run
# Add afternoon heating window: Eg: 12z (2 PM Local) | 15z (5 PM Local) | 18z (8 PM Local)
target_fxx = [12, 15, 18]   #Add correct times
results = []

for dt in tqdm(date_list, desc="Mining GFS Archives (City)"):
    date_str = dt.strftime('%Y-%m-%d')
    daily_temps = []

    for fxx in target_fxx:
        try:
            # Point Herbie exclusively to the 00:00 UTC initialization run
            H = Herbie(
                date_str,
                model='gfs',
                product='pgrb2.0p25',
                fxx=fxx,
                priority=['aws', 'nomads']
            )

            # Download ONLY the 2-meter temperature byte-range
            ds = H.xarray("TMP:2 m above ground")

            # Surgically extract the exact City pixel
            temp_kelvin = ds.t2m.sel(latitude=lat, longitude=lon_gfs, method='nearest').values

            # Convert Kelvin to Celsius
            daily_temps.append(float(temp_kelvin) - 273.15)

        except Exception as e:
            # Log missing data packets and keep the engine running
            continue

    # Calculate the Maximum predicted afternoon temperature for this day
    if daily_temps:
        max_predicted_temp = np.max(daily_temps)
        results.append({'Date': dt.date(), 'GFS_Predicted_High': round(max_predicted_temp, 2)})

    # AWS Politeness throttle
    time.sleep(0.5)


# Build the final DataFrame
df_gfs_city = pd.DataFrame(results)   # Add city name
df_gfs_city['Date'] = pd.to_datetime(df_gfs_city['Date']) #Add city name

print("\n==================================")
print("  City DATA ENGINEERING COMPLETE") #Add city name
print("==================================")
print(f"Successfully extracted {len(df_gfs_city)} days of pure 00z supercomputer forecasts.")
display(df_gfs_city.head())   #Add city name

In [ ]:
# 1. Convert the DataFrame to a CSV file
df_gfs_city.to_csv('city_gfs_checkpoint.csv', index=False)  #Add city name

# 2. Trigger the browser to download it to your actual computer
from google.colab import files
files.download('city_gfs_checkpoint.csv') #Add city name

print("Checkpoint downloaded successfully!")

In [ ]:
##loading the forecast data
df_gfs_city = pd.read_csv('/content/city_gfs_checkpoint.csv', parse_dates=['Date']) # Add City name
df_gfs_city.head()  #Add city name

ECMWF predictions

In [ ]:
# City Coordinates (Station)
lat = #Add latitude
lon_ecmwf = #Add lontitude. CRITICAL: ECMWF uses standard negative longitudes for West

# Define the 5-Year Window (Same as GFS)
utc_now = datetime.now(pytz.utc)
end_date = (utc_now - timedelta(days=2)).date()
start_date = end_date.replace(year=end_date.year - 5)
date_list = pd.date_range(start=start_date, end=end_date, freq='D')

# The Forecast Hours to pull from the 00z run (Same as GFS)
# Add correct hours: EG: 12z (2 PM Local) | 15z (5 PM Local) | 18z (8 PM Local)
target_fxx = [,, ]  #Add correct hours
results_ecmwf = []

for dt in tqdm(date_list, desc="Mining ECMWF Archives (City)"): # Add City name
    date_str = dt.strftime('%Y-%m-%d')
    daily_temps = []

    for fxx in target_fxx:
        try:
            # Point Herbie exclusively to the 00:00 UTC initialization run
            # 'ifs' = Integrated Forecast System (The European Supercomputer)
            H = Herbie(
                date_str,
                model='ifs',
                product='oper',
                fxx=fxx,
                priority=['azure', 'aws'] # Azure hosts the primary ECMWF open data bucket
            )

            # Download ONLY the 2-meter temperature byte-range
            ds = H.xarray(":2t:")

            # Surgically extract the exact city pixel #Add city name
            temp_kelvin = ds.t2m.sel(latitude=lat, longitude=lon_ecmwf, method='nearest').values

            # Convert Kelvin to Celsius
            daily_temps.append(float(temp_kelvin) - 273.15)

        except Exception as e:
            # Log missing data packets and keep the engine running
            continue

    # Calculate the Maximum predicted afternoon temperature for this day
    if daily_temps:
        max_predicted_temp = np.max(daily_temps)
        results_ecmwf.append({'Date': dt.date(), 'ECMWF_Predicted_High': round(max_predicted_temp, 2)})

    # Azure/AWS Politeness throttle
    time.sleep(0.5)

# Build the final DataFrame
df_ecmwf_city = pd.DataFrame(results_ecmwf) #Add city name
if not df_ecmwf_city.empty: #Add city name
    df_ecmwf_city['Date'] = pd.to_datetime(df_ecmwf_city['Date']) #Add city name

print("\n==================================")
print(" 🇪🇺 ECMWF DATA ENGINEERING COMPLETE")
print("==================================")
print(f"Successfully extracted {len(df_ecmwf_city)} days of pure 00z supercomputer forecasts.") #Add city name
display(df_ecmwf_city.head()) #Add city name

In [ ]:
# Convert the DataFrame to a CSV file
df_ecmwf_city.to_csv('ecmwf_city.csv', index=False)   #Add city name

# Trigger the browser to download it to your actual computer
from google.colab import files
files.download('ecmwf_city.csv')  # Add city name

print("Checkpoint downloaded successfully!")

In [ ]:
##loading the forecast data
df_ecmwf_city = pd.read_csv('/content/ecmwf_madrid.csv', parse_dates=['Date'])  #Add city name
df_ecmwf_city.head()  #Add city name

GFS 850hPa temps

In [ ]:
## City Coordinates (Station)
lat = # Add latitude
lon_gfs =  #Add lontitude: we use the 360 Longitude Math

# Define the 5-Year Window
# 1. Anchor strictly to UTC
utc_now = datetime.now(pytz.utc)
# 2. Subtract 2 days for the AWS Government indexing buffer
end_date = (utc_now - timedelta(days=2)).date()
# 3. Look back exactly 5 years for Compute Optimization
start_date = end_date.replace(year=end_date.year - 5)
date_list = pd.date_range(start=start_date, end=end_date, freq='D')

# The Forecast Hours to pull from the 00z run
# Add correct hours
target_fxx = [,]  #Add correct hours eg 12z and 15z
results = []

for dt in tqdm(date_list, desc="Mining GFS 850hPa Archives (City)"):  #Add city name
    date_str = dt.strftime('%Y-%m-%d')
    daily_temps = []

    for fxx in target_fxx:
        try:
            # Point Herbie exclusively to the 00:00 UTC initialization run
            H = Herbie(
                date_str,
                model='gfs',
                product='pgrb2.0p25',
                fxx=fxx,
                priority=['aws', 'nomads']
            )

            # Download ONLY the 850 millibar temperature byte-range
            ds = H.xarray("TMP:850 mb")

            # Surgically extract the exact City pixel (Upper air temp is 't', not 't2m')  #Add city name
            temp_kelvin = ds.t.sel(latitude=lat, longitude=lon_gfs, method='nearest').values
            # Convert Kelvin to Celsius
            daily_temps.append(float(temp_kelvin) - 273.15)

        except Exception as e:
            # Log missing data packets and keep the engine running
            continue
        # Calculate the Maximum predicted afternoon temperature for the 850hPa level
    if daily_temps:
        max_850_temp = np.max(daily_temps)
        results.append({'Date': dt.date(), 'GFS_850hPa_Temp': round(max_850_temp, 2)})

    # AWS Politeness throttle
    time.sleep(0.5)

# Build the final DataFrame
df_gfs_850 = pd.DataFrame(results)
df_gfs_850['Date'] = pd.to_datetime(df_gfs_850['Date'])

print("\n==================================")
print("  CITY 850hPa ENGINEERING COMPLETE") #Add city name
print("==================================")
print(f"Successfully extracted {len(df_gfs_850)} days of Upper Air data.")
display(df_gfs_850.head())

In [ ]:
# 1. Convert the DataFrame to a CSV file
df_gfs_850.to_csv('city_gfs_850_checkpoint.csv', index=False) #Add city name

# 2. Trigger the browser to download it to your actual computer
from google.colab import files
files.download('city_gfs_850_checkpoint.csv') #Add city name

print("Checkpoint downloaded successfully!")

In [ ]:
#loading the file
df_gfs_city_850 = pd.read_csv('/content/city_gfs_850_checkpoint.csv', parse_dates=['Date']) #Add city name
df_gfs_city_850.head()  #Add city name

Open-Meteo" Alpha API: Solar Radiation and Soil Thermodynamics historically and live via the Open-Meteo API

In [ ]:
!pip install openmeteo-requests requests-cache retry-requests
import openmeteo_requests
import requests_cache
from retry_requests import retry
import pandas as pd

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

#specifying the period
utc_now = datetime.now(pytz.utc)
end_date = (utc_now - timedelta(days=2)).date()
start_date = end_date.replace(year=end_date.year - 5)

# We want X AM local data (which is 08:00 UTC)
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": #Add latitude,
	"longitude": #Add lontitude (eg: -3,78),
	"start_date": start_date.strftime('%Y-%m-%d'), # Converts to "YYYY-MM-DD"
	"end_date": end_date.strftime('%Y-%m-%d'),     # Converts to "YYYY-MM-DD"
	"hourly": ["direct_normal_irradiance", "soil_temperature_0_to_7cm", "soil_moisture_0_to_7cm","diffuse_radiation"],
	"timezone": "Continent/City"	#Add continent/city
}

responses = openmeteo.weather_api(url, params=params)
response = responses[0]
hourly = response.Hourly()


#Process data
hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["DNI_Radiation_Wm2"] = hourly.Variables(0).ValuesAsNumpy()
hourly_data["Soil_Temp_7cm"] = hourly.Variables(1).ValuesAsNumpy()
hourly_data["Soil_Moisture_7cm"] = hourly.Variables(2).ValuesAsNumpy()
hourly_data["DHI_Radiation_Wm2"] = hourly.Variables(3).ValuesAsNumpy()

df_thermo = pd.DataFrame(hourly_data)

# We only care about the exact state at 10:00 AM Local Time
df_thermo['date'] = df_thermo['date'].dt.tz_convert('Europe/Madrid')	#Add continent/city name
df_thermo_10am = df_thermo[df_thermo['date'].dt.hour == 10].copy()

# Create a clean 'Date' column to merge with your Super Brain
df_thermo_10am['Date'] = df_thermo_10am['date'].dt.date
df_thermo_10am.drop(columns=['date'], inplace=True)

print("✅ High-Resolution Thermodynamics Downloaded.")
display(df_thermo_10am.head())



Data Engeneering

In [ ]:
print("Converting raw data from Fahrenheit to Celsius...")

# Convert Air Temperature
df_raw['Temp_C'] = (df_raw['Temp_F'] - 32) * (5.0 / 9.0)

# Force 'alti' to be numeric, turning errors/blanks into NaN
df_raw['alti'] = pd.to_numeric(df_raw['alti'], errors='coerce')

# Optional: Forward-fill any tiny gaps in the pressure data (pressure moves slowly)
df_raw['alti'] = df_raw['alti'].ffill()

# Convert Dew Point (Moisture)
df_raw['Dew_Point_C'] = (df_raw['Dew_Point_F'] - 32) * (5.0 / 9.0)

# Drop the old Fahrenheit columns so we don't accidentally use them
df_raw = df_raw.drop(columns=['Temp_F', 'Dew_Point_F'])

print("Conversion complete. Here is your metric DataFrame:")
display(df_raw.head())


In [ ]:
#taking a subset of our data.
#Target hour 10 AM.
#calculate derivatives of temprature, dewpoint and overnight lowest temprature
#Add day number 1-366
#perform cyclical encoding in the wind_dir_deg



# Ensure Timestamp is the index and the data is sorted chronologically
if 'Timestamp' in df_raw.columns:
    df_raw.set_index('Timestamp', inplace=True)
df_raw.sort_index(inplace=True)



#  Target (y): Max Temp between 12:00 AM and 20:00 PM
afternoon_df = df_raw.between_time('12:00', '20:00')
daily_max = afternoon_df.groupby(afternoon_df.index.date)['Temp_C'].max().reset_index()
daily_max.columns = ['Date', 'Actual_Max_Temp']


#  Base Snapshot (10:00 AM)
df_10am = df_raw.between_time('09:45', '10:15').groupby(df_raw.between_time('09:45', '10:15').index.date).last().reset_index()

# Rename columns to  ML feature names
df_10am.rename(columns={
    'index': 'Date',
    'Temp_C': 'Temp_10AM',
    'Dew_Point_C': 'Dew_Point_10AM',
    'Wind_Dir_Deg': 'Wind_Dir_10AM',
    'Wind_Speed_Kts': 'Wind_Speed_10AM',
    'Cloud_Cover_Okta': 'Cloud_Cover_10AM',
    'alti': 'alti_10AM'
}, inplace=True)

# Safely merge  24h Rain into the 10AM snapshot
df_10am =df_10am
df_10am.head()


# Momentum Snapshot (7:00 AM) for Derivatives
df_7am = df_raw.between_time('06:45', '07:15').groupby(df_raw.between_time('06:45', '07:15').index.date).last().reset_index()
df_7am.rename(columns={'index': 'Date', 'Temp_C': 'Temp_7AM', 'Dew_Point_C': 'Dew_Point_7AM','alti': 'alti_7AM'}, inplace=True)

# Overnight Low (Midnight to 6:00 AM)
df_overnight = df_raw.between_time('00:00', '06:00')
overnight_lows = df_overnight.groupby(df_overnight.index.date)['Temp_C'].min().reset_index()
overnight_lows.columns = ['Date', 'Overnight_Low']

#  Merge everything together securely
df_train = pd.merge(df_10am, daily_max, on='Date', how='inner')
df_train = pd.merge(df_train, df_7am[['Date', 'Temp_7AM', 'Dew_Point_7AM', 'alti_7AM']], on='Date', how='inner')
df_train = pd.merge(df_train, overnight_lows, on='Date', how='inner')
df_train.head()


# Calculate Derivatives (The Momentum)
df_train['Temp_Change_3hr'] = df_train['Temp_10AM'] - df_train['Temp_7AM']
df_train['Dew_Point_Change_3hr'] = df_train['Dew_Point_10AM'] - df_train['Dew_Point_7AM']
df_train['Pressure_Change_3hr'] = df_train['alti_10AM'] - df_train['alti_7AM']

# If Temp is going up and Dew Point is going down, this number explodes.
df_train['Heat_Momentum_Ratio'] = df_train['Temp_Change_3hr'] - df_train['Dew_Point_Change_3hr']

# Calculate Relative Humidity (More stable than VPD for MAE)
df_train['RH_10AM'] = 100 * (np.exp((17.625 * df_train['Dew_Point_10AM']) / (243.04 + df_train['Dew_Point_10AM'])) /
                           np.exp((17.625 * df_train['Temp_10AM']) / (243.04 + df_train['Temp_10AM'])))

# Morning Heat Gap (How much has it already heated since the floor?)
df_train['Early_Heating_Effort'] = df_train['Temp_10AM'] - df_train['Overnight_Low']

# Cyclical Encoding (Wind & Day of Year)
df_train['Date'] = pd.to_datetime(df_train['Date'])
df_train['Day_of_Year'] = df_train['Date'].dt.dayofyear

# Wind Direction (Circle of 360 degrees)
df_train['Wind_Sin'] = np.sin(df_train['Wind_Dir_10AM'] * (2. * np.pi / 360))
df_train['Wind_Cos'] = np.cos(df_train['Wind_Dir_10AM'] * (2. * np.pi / 360))

# Day of the Year (Circle of 366 degrees for Leap Year safety)
df_train['Day_Sin'] = np.sin(df_train['Day_of_Year'] * (2. * np.pi / 366))
df_train['Day_Cos'] = np.cos(df_train['Day_of_Year'] * (2. * np.pi / 366))

# Clean up raw/confusing columns to prevent overfitting
df_train = df_train.drop(columns=[
    'Wind_Dir_10AM', 'Day_of_Year', 'Temp_7AM', 'Dew_Point_7AM', 'Timestamp','alti_7AM'
], errors='ignore')

print("\n--- Final 'Full Physics' ML DataFrame Ready ---")
print(f"Total Usable Trading Days: {len(df_train):,}")
display(df_train.head())


In [ ]:
#Merging the two datasets: df_train and df_gfs_city and df_gfs_city_850   #Add city name

# 1. Force all 'Date' columns to be pure Pandas Datetime objects
df_train['Date'] = pd.to_datetime(df_train['Date'])
df_gfs_city['Date'] = pd.to_datetime(df_gfs_city['Date'])   ##Add city name
df_gfs_city_850['Date'] = pd.to_datetime(df_gfs_city_850['Date']) #Add city name
df_thermo_10am['Date'] = pd.to_datetime(df_thermo_10am['Date'])
df_ecmwf_city['Date'] = pd.to_datetime(df_ecmwf_city['Date'])   #Add city name

#  Merge
df_super_brain = (
    df_train
    .merge(df_gfs_city, on='Date', how='inner') #Add city name
    .merge(df_gfs_city_850, on='Date', how='inner') #Add city name
    .merge(df_thermo_10am, on='Date', how='inner')
    .merge(df_ecmwf_city, on='Date', how='inner') #Add city name
)

# Clean up the index
df_super_brain = df_super_brain.sort_values('Date').reset_index(drop=True)

# Sort chronologically just to be safe
df_super_brain = df_super_brain.sort_values('Date').reset_index(drop=True)

# After merging df_super_brain in Cell 11:
df_super_brain['GFS_Heat_Gap'] = df_super_brain['GFS_Predicted_High'] - df_super_brain['Temp_10AM']

# The Chaos Detector: How violently do the models disagree?
df_super_brain['Supercomputer_Spread'] = np.abs(df_super_brain['GFS_Predicted_High'] - df_super_brain['ECMWF_Predicted_High'])

# The Bias Direction: Positive means GFS is hotter, Negative means ECMWF is hotter
df_super_brain['Model_Bias_Direction'] = df_super_brain['GFS_Predicted_High'] - df_super_brain['ECMWF_Predicted_High']

# The Blended Supercomputer Baseline
df_super_brain['Consensus_Predicted_High'] = (df_super_brain['GFS_Predicted_High'] + df_super_brain['ECMWF_Predicted_High']) / 2

#  The 'Oven' Factor: How extreme is the GFS surface prediction compared to the upper air reality?
df_super_brain['Forecasted_Lapse_Rate'] = df_super_brain['GFS_Predicted_High'] - df_super_brain['GFS_850hPa_Temp']

# If this ratio is high, it means the sun is blocked, but the light is trapped in the clouds.
# It prevents the AI from falsely predicting a 'cool' day when DNI drops.
df_super_brain['Scattering_Ratio'] = df_super_brain['DHI_Radiation_Wm2'] / (df_super_brain['DNI_Radiation_Wm2'] + 1)

# Also, ensure we have the absolute pressure
df_super_brain['Pressure_10AM'] = df_super_brain['alti_10AM']

# The Thermal Updraft: How much hotter is the 'Stove' than the 'Air'?
# A massive positive gap means explosive afternoon heating.
df_super_brain['Ground_to_Air_Gradient'] = df_super_brain['Soil_Temp_7cm'] - df_super_brain['Temp_10AM']

#  Safety Check
# If Herbie missed a packet and recorded a NaN, we drop that day
df_super_brain.dropna(subset=['GFS_Predicted_High'], inplace=True)


# Re-order the columns so the "Target" is easy to see at the end
# Assuming your target column from earlier is still named 'Actual_Max_Temp'
cols = [c for c in df_super_brain.columns if c not in ['Date', 'Actual_Max_Temp']]
df_super_brain = df_super_brain[['Date'] + cols + ['Actual_Max_Temp']]

# Calculate how much the 'World' was wrong yesterday
# shift by 1 day to ensure no data leakage
df_super_brain['Yesterday_Bias'] = (df_super_brain['Actual_Max_Temp'] - df_super_brain['GFS_Predicted_High']).shift(1)

# Heat Persistence: Was yesterday hotter than the day before?
df_super_brain['Heat_Trend'] = df_super_brain['Actual_Max_Temp'].shift(1) - df_super_brain['Actual_Max_Temp'].shift(2)

# 3-Day Rolling Bias
# This captures persistent soil-moisture or air-mass errors
df_super_brain['Bias_3Day_Mean'] = df_super_brain['Yesterday_Bias'].rolling(window=3).mean()

# The Advection Alpha: How hard is the mountain/urban air blowing?
# Positive = Strong Cold Front from the North
# Negative = Strong Heat Wave from the South
df_super_brain['Cooling_Advection'] = df_super_brain['Wind_Speed_10AM'] * df_super_brain['Wind_Cos']

#  Morning Delta (Today vs Yesterday)
# If today at 10 AM is 2 degrees hotter than yesterday at 10 AM,
# the 'High' is almost guaranteed to be higher.
df_super_brain['Temp_10AM_vs_Yesterday'] = df_super_brain['Temp_10AM'] - df_super_brain['Temp_10AM'].shift(1)

#If the wind is high, the gap is harder to close.
df_super_brain['Gap_Wind_Resistance'] = df_super_brain['GFS_Heat_Gap'] / (df_super_brain['Wind_Speed_10AM'] + 1)

# removes the NaNs
df_super_brain = df_super_brain.dropna(subset=['Yesterday_Bias', 'Heat_Trend'])
# Clean up the new NaNs from the 3-day window
df_super_brain = df_super_brain.dropna(subset=['Bias_3Day_Mean', 'Temp_10AM_vs_Yesterday'])



print("\n========================================")
print("  SUPER-BRAIN DATASET FULLY ASSEMBLED")
print("========================================")
print(f"Total Trading Days: {len(df_super_brain):,}")
display(df_super_brain.head())
df_super_brain.columns

In [ ]:
###diagnostic cell. Compare the max temps with the max temps from the wonderground website
# Load the new Oracle data you just uploaded to Colab
df_oracle = pd.read_csv("LEMD_FINAL_ORACLE_DATA_City.csv")  #Add city name

# Standardize dates to ensure a perfect, bug-free merge
df_oracle['Date'] = pd.to_datetime(df_oracle['Date'])
df_super_brain['Date'] = pd.to_datetime(df_super_brain['Date'])

# Merge the Oracle answers into your existing DataFrame
df_compare = pd.merge(df_super_brain, df_oracle, on='Date', how='inner')

# Calculate the discrepancy
df_compare['Oracle_Distortion'] = df_compare['Oracle_High_C'] - df_compare['Actual_Max_Temp']


# Extract the Statistics
total_days = len(df_compare)

# Check for missing data first
failed_scrapes = df_compare['Oracle_Distortion'].isna().sum()

# We define a "Perfect Match" as anything within 0.05 degrees
perfect_matches = len(df_compare[df_compare['Oracle_Distortion'].abs() <= 0.05])

oracle_lower = len(df_compare[df_compare['Oracle_Distortion'] < -0.05])
oracle_higher = len(df_compare[df_compare['Oracle_Distortion'] > 0.05])

print(f"\n COMPARISON METRICS (Over {total_days} days):")
print(f" 'Perfect' Matches (Within 0.05°C): {perfect_matches} days ({(perfect_matches/total_days)*100:.1f}%)")
print(f" Oracle was LOWER (Deleted Peaks): {oracle_lower} days ({(oracle_lower/total_days)*100:.1f}%)")
print(f" Oracle was HIGHER (Rounding Traps): {oracle_higher} days ({(oracle_higher/total_days)*100:.1f}%)")
print(f" Missing IBM Data (Failed Scrapes): {failed_scrapes} days ({(failed_scrapes/total_days)*100:.1f}%)")

# sanity check
total_accounted = perfect_matches + oracle_lower + oracle_higher + failed_scrapes
print(f"Audit Check: {total_accounted}/{total_days} days accounted for.")

# Show a sample of days where the Oracle was completely wrong
print("\n 5 WORST DOWNWARD GLITCHES:")
worst_days = df_compare.sort_values(by='Oracle_Distortion').tail(5)
print(worst_days[['Date', 'Actual_Max_Temp', 'Oracle_High_C', 'Oracle_Distortion']].to_string(index=False))

# OVERWRITE THE OLD TARGET
df_super_brain = df_compare.copy()

df_super_brain.head()

XGBOOST model training

In [ ]:

#Calculate target value
df_super_brain['Heating_Delta'] = df_super_brain['Oracle_High_C'] - df_super_brain['Temp_10AM']
y = df_super_brain['Heating_Delta']

# Define Features (X)
X = df_super_brain.drop(columns=['Date', 'Actual_Max_Temp', 'station', 'Heating_Delta','Oracle_High_C','Oracle_Distortion','alti_10AM','GFS_850hPa_Temp',
'GFS_Predicted_High','ECMWF_Predicted_High'])

#Chronological Train/Test Split (80% Train, 20% Test)
split_index = int(len(df_super_brain) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Training on {len(X_train)} days (The Past)...")
print(f"Testing on {len(X_test)} days (The Future)...")

X.columns

In [ ]:
from sklearn.utils.extmath import randomized_svd
#defining the timeseries split
#We use 3 fold for the cross validation
tscv = TimeSeriesSplit(n_splits=3)

# Define the Hyperparameter Grid
# The computer will test every possible combination of these numbers
param_grid = {'n_estimators':[300,800,1000], 'learning_rate':[0.005,0.05,0.1],'max_depth':[3,4,5,7],'subsample':[0.8,1.0]}

#XGBOOST model initilization
base_model = xgb.XGBRegressor(objective='reg:absoluteerror',colsample_bytree=0.8,tree_method='hist',random_state=123)


# Set up the Grid Search
print("Commencing Hyperparameter Grid Search. This will train 162 models.")
print("Please wait, this may take 1-3 minutes...")

grid_search = GridSearchCV(estimator=base_model,param_grid=param_grid, cv = tscv, scoring = 'neg_mean_absolute_error',verbose=1,n_jobs=-1)

# Fit the Grid Search on the Training Data
grid_search.fit(X_train,y_train)

#Extract the absolute best model
best_model = grid_search.best_estimator_

print("\n==================================")
print("     OPTIMIZATION COMPLETE        ")
print("==================================")
print("The Best Hyperparameters for City are:") #Add city name
for param, value in grid_search.best_params_.items():
    print(f" - {param}: {value}")

#Final Test: Evaluate the tuned model on the unseen Future (X_test)
print("\nEvaluating Optimized Model on the Future (Test Set)...")
delta_predictions = best_model.predict(X_test)
actual_predicted_highs = X_test['Temp_10AM'] + delta_predictions
actual_real_highs = y_test + X_test['Temp_10AM']

final_mae = mean_absolute_error(actual_real_highs, actual_predicted_highs)
print(f"🔥 True Heating Delta MAE: {final_mae:.2f}°C")
#Plotting features importace
plt.figure(figsize=(10, 6))
xgb.plot_importance(best_model, importance_type='weight', max_num_features=10,
                    height=0.5, show_values=False, color='#3498db')
plt.title('What drives the Alpha? (Feature Importance)')
plt.grid(False)
plt.tight_layout()
plt.show()

Downloading the model

In [ ]:
from google.colab import files

best_model.save_model('city_MAE_###_oracle.json') #Add city name and MAE value

#Trigger the browser to download the file to your hard drive
files.download('city_MAE_###_oracle.json')

print("Model successfully packaged and downloaded!")